In [1]:
import numpy as np
import struct
from array import array
from os.path  import join

%matplotlib inline
import random
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader

In [2]:
from data import build_dataloaders
from model import ConvModel
from util import test_loss, test_accuracy

In [7]:
cross_entropy = nn.CrossEntropyLoss()

In [8]:
# Just find out how many params
# We will re-initialize the model in-loop for reproducibility

model = ConvModel()
sum([p.numel() for p in model.parameters()])

66954

In [9]:
# training loop

n_batches = 1000

for lr in [0.001, 0.01, 0.1]:


    torch.manual_seed(1)
    model = ConvModel()
    
    train_loader, test_loader = build_dataloaders(seed=1)
    
    i = 0
    while i < n_batches:
        for x, y in train_loader:
            y_pred = model(x)
            loss = cross_entropy(y_pred, y)
            if i % 250 == 0 or i == n_batches - 1:
                print(f"Loss at step {i:4} is {loss.item():0.3f}")
                print(f"Test loss at step {i:4} is {test_loss(model, test_loader):0.3f}")
        
            loss.backward()
    
            with torch.no_grad():
                for p in model.parameters():
                    p -= lr * p.grad
            model.zero_grad()
            
            i += 1
            if i == n_batches:
                break

    print(f"Learning rate: {lr}; test accuracy: {test_accuracy(model, test_loader)}")


Loss at step    0 is 2.435
Computed test loss over 10000 items
Test loss at step    0 is 2.306
Computed test accuracy over 10000 items
Learning rate: 0.001; test accuracy: 0.1009
Computed test accuracy over 10000 items
Learning rate: 0.001; test accuracy: 0.101
Computed test accuracy over 10000 items
Learning rate: 0.001; test accuracy: 0.101
Computed test accuracy over 10000 items
Learning rate: 0.001; test accuracy: 0.101
Computed test accuracy over 10000 items
Learning rate: 0.001; test accuracy: 0.101
Computed test accuracy over 10000 items
Learning rate: 0.001; test accuracy: 0.101
Computed test accuracy over 10000 items
Learning rate: 0.001; test accuracy: 0.101
Computed test accuracy over 10000 items
Learning rate: 0.001; test accuracy: 0.101
Computed test accuracy over 10000 items
Learning rate: 0.001; test accuracy: 0.101
Computed test accuracy over 10000 items
Learning rate: 0.001; test accuracy: 0.101
Computed test accuracy over 10000 items
Learning rate: 0.001; test accurac

KeyboardInterrupt: 

In [ ]:
# custom optimizer
# that only uses the sign (+ or -) of the gradient, not its value
class GradSignOptimizer:
    def __init__(self, named_params, *, lr):
        self.named_params = [item for item in named_params]

        # initialize counts
        self.grad_counts = {}
        for n, p in self.named_params:
            self.grad_counts[n] = torch.randint(-64, 64, p.shape, dtype=torch.int8)
        

    def step(self):
        for n, p in self.named_params:
            new_count = 2 * (p.grad > 0) - 1
            self.grad_counts[n] -= (self.grad_counts[n] + 4) // 8
            self.grad_counts[n] += 8 * new_count     # max val will be +- 64 or so

            p.data -= lr * (self.grad_counts[n] / 64.0)

    def zero_grad(self):
        for n, p in self.named_params:
            p.grad = None

In [ ]:
torch.manual_seed(1)
model = ConvModel()


In [ ]:
# training loop

n_batches = 100 # 2000
cross_entropy = nn.CrossEntropyLoss()
lr = 0.001


optim = GradSignOptimizer(model.named_parameters(), lr=lr)

In [ ]:

model.train()


train_loader, test_loader = build_dataloaders(seed=1)
print(next(iter(train_loader))[1])

i = 0
while i < n_batches:
    for x, y in train_loader:
        y_pred = model(x)
        loss = cross_entropy(y_pred, y)
        if i % 250 == 0 or i == n_batches - 1:
            print(f"Loss at step {i:4} is {loss.item():0.3f}")
            print(f"Test loss at step {i:4} is {test_loss(model, test_loader):0.3f}")
    
        optim.zero_grad()
        loss.backward()
        optim.step()

        
        i += 1
        if i == n_batches:
            break




In [ ]:
test_accuracy(model, test_loader)